# Segmenting remote sensing imagery with text prompts and the Segment Anything Model (SAM)

[![image](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/opengeos/segment-geospatial/blob/main/docs/examples/text_prompts.ipynb)
[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/segment-geospatial/blob/main/docs/examples/text_prompts.ipynb)

This notebook shows how to generate object masks from text prompts with the Segment Anything Model (SAM).

Make sure you use GPU runtime for this notebook. For Google Colab, go to `Runtime` -> `Change runtime type` and select `GPU` as the hardware accelerator.

## Install dependencies

Uncomment and run the following cell to install the required dependencies.

In [ ]:
%pip install segment-geospatial groundingdino-py leafmap localtileserver

In [ ]:
import leafmap
from samgeo.common import tms_to_geotiff
from samgeo.text_sam import LangSAM
import os
import json
from google.colab import drive
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from samgeo.text_sam import LangSAM
import pandas as pd

## Create an interactive map

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install shapely wandb load_dotenv
import sys
sys.path.append('/content/drive/My Drive')   

from eval import GeoNLIEvaluator   

print("Initializing GeoNLI Evaluator...")
evaluator = GeoNLIEvaluator()


In [ ]:
import os
from PIL import Image

def convert_png_to_tif(input_dir, output_dir):
    """
    Converts all .png images in the input directory to .tif (TIFF) format
    and saves them in the output directory.
    """
    if not os.path.exists(input_dir):
        print(f"Error: Input directory not found: {input_dir}")
        return

    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output directory ensured: {output_dir}")

    conversion_count = 0

    # Iterate over all files in the input directory
    for filename in os.listdir(input_dir):
        if filename.lower().endswith('.png'):
            # Construct full file paths
            input_path = os.path.join(input_dir, filename)

            # Create the output filename by replacing .png with .tif
            base_name = os.path.splitext(filename)[0]
            output_filename = base_name + '.tif'
            output_path = os.path.join(output_dir, output_filename)

            try:
                # Open the PNG image
                img = Image.open(input_path)

                # Save as TIFF. The 'tiff' format automatically handles
                # saving based on the standard PIL format definitions.
                img.save(output_path, format='tiff')

                conversion_count += 1

            except Exception as e:
                print(f"Failed to convert {filename}: {e}")

    print(f"\nConversion complete. Total files processed: {conversion_count}")
    if conversion_count == 0:
         print(f"Note: Found no .png files in the input directory: {input_dir}")


# --- CONFIGURATION (UPDATE THESE PATHS) ---
# Assuming these directories are inside your Google Drive mount point
# You can adjust these to your exact paths
BASE_DIR = "/content/drive/My Drive/"
INPUT_DIR_NAME = "Images_Val_Drive"
OUTPUT_DIR_NAME = "Images_val_tif_Drive"

INPUT_PATH = os.path.join(BASE_DIR, INPUT_DIR_NAME)
OUTPUT_PATH = os.path.join(BASE_DIR, OUTPUT_DIR_NAME)
# ------------------------------------------


# --- EXECUTION ---
# Ensure Google Drive is mounted before attempting conversion
# Note: The 'drive.mount' block from your previous script must run successfully
# before executing this function.
if os.path.exists("/content/drive"):
    convert_png_to_tif(INPUT_PATH, OUTPUT_PATH)
else:
    print("Google Drive is not mounted. Please run the drive.mount() command first.")

In [ ]:
from PIL import Image
import os

def convert_png_to_tif(input_path, output_name):
    """
    Converts an image file (e.g., PNG) to TIFF format.

    Args:
        input_path (str): The full path to the source image file (e.g., 'path/to/destination.png').
        output_name (str): The desired name of the output TIFF file (e.g., 'img_123.tif').
    """
    try:
        # 1. Open the image
        img = Image.open(input_path)

        # 2. Define the output path
        # It places the output file in the same directory as the input file
        output_dir = os.path.dirname(input_path)
        output_path = os.path.join(output_dir, output_name)

        # 3. Save the image in TIFF format
        # Pillow automatically determines the format from the file extension (.tif)
        img.save(output_path, format="TIFF")

        print(f"Conversion successful!")
        print(f"Input: {input_path}")
        print(f"Output: {output_path}")

    except FileNotFoundError:
        print(f"ERROR: Input file not found at {input_path}")
    except Exception as e:
        print(f"An error occurred during conversion: {e}")

# --- Set your file paths here ---
# Replace 'path/to/destination.png' with the actual path to your source PNG file
input_file_path = "/content/drive/My Drive/05863_0000.png"

# Replace 'img_123.tif' with your desired output name
output_file_name = "Image1.tif"
# --------------------------------

# Execute the function
convert_png_to_tif(input_file_path, output_file_name)

In [ ]:
from samgeo.text_sam import LangSAM
import os
import matplotlib.pyplot as plt
from google.colab import drive
import json

# --- Mount Google Drive ---
try:
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting drive: {e}.")
    exit()

# --- CONFIGURATION ---
IMAGE_PATH = "/content/drive/My Drive/Image1.tif"
TEXT_PROMPT = "The solitary windmill in the image is positioned towards the right edge of the frame."
JSON_PATH = "/content/drive/My Drive/05863_0000.json"
OUTPUT_MASK = "/content/drive/My Drive/segmented_mask.tif"
OUTPUT_VECTOR = "/content/drive/My Drive/segmented_objects.gpkg"
# ---------------------

# Check existence
if not os.path.exists(IMAGE_PATH):
    print(f"Error: The image file '{IMAGE_PATH}' was not found.")
else:
    print(f"Successfully located custom image: {IMAGE_PATH}")

    # 1. Initialize LangSAM class
    print("\nInitializing LangSAM Model...")
    sam = LangSAM()

    # 2. Segment the image
    print(f"Starting segmentation for prompt: '{TEXT_PROMPT}'")
    sam.predict(
        IMAGE_PATH,
        TEXT_PROMPT,
        box_threshold=0.24,
        text_threshold=0.24
    )

    if sam.boxes is not None:
      boxes_np = sam.boxes.cpu().numpy()
      print("\n--- Bounding Box Coordinates (Pixel Format: [x_min, y_min, x_max, y_max]) ---")
      print(f"Found {len(boxes_np)} objects matching '{TEXT_PROMPT}':")
      for i, box in enumerate(boxes_np):
        print(f"Object {i+1}: {box}")
      print("--------------------------------------------------------------------------")
    else:
      print(f"\nNo objects found for the prompt: '{TEXT_PROMPT}'")

    print("\n--- A. Extracting Ground Truth Box ---")
    with open(JSON_PATH, 'r') as f:
        data = json.load(f)
    gt_coords_norm = data['objects'][0]['obj_coord']
    print(f"Ground Truth Box (Normalized): {gt_coords_norm}")

    img = Image.open(IMAGE_PATH)
    img_width, img_height = img.size
    fig, ax = plt.subplots(1, figsize=(12, 12))
    ax.set_title(f"Comparison: Predicted (Blue) vs. Ground Truth (Red) for '{TEXT_PROMPT}'")

    # Helper function to convert normalized [x1, y1, x2, y2] to Matplotlib [x, y, width, height]
    def denormalize_box(norm_coords, width, height):
        x_min, y_min, x_max, y_max = norm_coords
        x = x_min * width
        y = y_min * height
        w = (x_max - x_min) * width
        h = (y_max - y_min) * height
        return (x, y, w, h)

    # 1. Plot GROUND TRUTH BOX (Red)
    gt_x, gt_y, gt_w, gt_h = denormalize_box(gt_coords_norm, img_width, img_height)
    gt_rect = patches.Rectangle((gt_x, gt_y), gt_w, gt_h,
                                linewidth=3, edgecolor='red',
                                facecolor='none', label='Ground Truth')
    ax.add_patch(gt_rect)
    plt.legend()
    plt.axis('off')
    plt.show()

    sam.show_anns( cmap="Greens", add_boxes=True, alpha=0.7, title=f"Segmentation of {TEXT_PROMPT.title()}", output=OUTPUT_MASK, blend=False, )


In [ ]:
from samgeo.text_sam import LangSAM
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches # NEW: Added for plotting rectangles
from google.colab import drive
import torch
import numpy as np
from PIL import Image # NEW: Added for reading image dimensions
import json # NEW: Added for reading ground truth JSON

# --- CONFIGURATION ---
# IMAGE_PATH = "/content/drive/My Drive/Image1.tif"
IMAGE_PATH = "/content/drive/My Drive/Images_val_tif/P0837_0001.tif"
JSON_PATH = "/content/drive/My Drive/P0837_0001.json"

with open(JSON_PATH, 'r') as f:
  data = json.load(f)

TEXT_PROMPT = object_class = data['objects'][0]['referring_sentence']

# ---------------------

# Check existence
if not os.path.exists(IMAGE_PATH):
    print(f"Error: The image file '{IMAGE_PATH}' was not found.")
else:
    print(f"Successfully located custom image: {IMAGE_PATH}")

    # 1. Initialize LangSAM class
    print("\nInitializing LangSAM Model. This may take a moment to download weights...")
    sam = LangSAM()

    # 2. Segment the image
    print(f"Starting segmentation for prompt: '{TEXT_PROMPT}'")
    sam.predict(
        IMAGE_PATH,
        TEXT_PROMPT,
        box_threshold=0.24,
        text_threshold=0.24
    )

    # 3. --- EXTRACT AND PRINT BOUNDING BOXES ---
    if sam.boxes is not None:
        # Move tensor to CPU and convert to NumPy array for printing
        boxes_np = sam.boxes.cpu().numpy()
        print("\n--- Bounding Box Coordinates (Pixel Format: [x_min, y_min, x_max, y_max]) ---")
        print(f"Found {len(boxes_np)} objects matching '{TEXT_PROMPT}':")
        for i, box in enumerate(boxes_np):
            print(f"Object {i+1} (Predicted): {box}")
        print("--------------------------------------------------------------------------")
    else:
        print(f"\nNo objects found for the prompt: '{TEXT_PROMPT}'")

    # Helper function for denormalization (for Ground Truth box)
    def denormalize_box(norm_coords, width, height):
        x_min, y_min, x_max, y_max = norm_coords
        x = x_min * width
        y = y_min * height
        w = (x_max - x_min) * width
        h = (y_max - y_min) * height
        return (x, y, w, h)

    sam.show_anns(
        cmap="Greens",
        add_boxes=True,
        alpha=0.7,
        blend=True,
    )

    try:
        with open(JSON_PATH, 'r') as f:
            data = json.load(f)
        gt_coords_norm = data['objects'][0]['obj_coord']
        print(f"Ground Truth Box (Normalized): {gt_coords_norm}")

        img = Image.open(IMAGE_PATH)
        img_width, img_height = img.size

        gt_x, gt_y, gt_w, gt_h = denormalize_box(gt_coords_norm, img_width, img_height)
        gt_rect = patches.Rectangle((gt_x, gt_y), gt_w, gt_h,
                                    linewidth=3, edgecolor='red',
                                    facecolor='none', label='Ground Truth')
        ax.add_patch(gt_rect)

        # Adjust legend to show both predicted boxes (from sam.show_anns) and GT
        ax.legend(handles=[gt_rect], loc='lower right')

    except FileNotFoundError:
        print(f"Warning: Ground truth JSON file not found at {JSON_PATH}. Skipping GT overlay.")
    except Exception as e:
        print(f"Warning: Failed to parse or plot Ground Truth box: {e}")

    plt.show() # Only show call in the script

    print(f"\nProcessing finished.")

    boxes_np = sam.boxes.cpu().numpy()


In [ ]:
def plot_boxes_comparison_revised_fixed(boxes_np):
    print("\n--- A. Extracting Ground Truth Box ---")
    with open(JSON_PATH, 'r') as f:
        data = json.load(f)
    gt_coords_norm = data['objects'][0]['obj_coord']
    print(f"Ground Truth Box (Normalized): {gt_coords_norm}")

    # --- B. GET PREDICTED BOX from LangSAM (FINAL REVISED EXTRACTION) ---
    print("\n--- B. Generating Predicted Box (LangSAM) ---")


    try:
        for i, box in enumerate(boxes_np):
            print(f"Object {i+1} (Predicted): {box}")
        print("--------------------------------------------------------------------------")

    except Exception as e:
        print(f"ERROR: Failed to extract predicted box data after prediction: {e}")
        # Print the attributes available on the sam object for debugging
        print("\n--- DEBUG: sam attributes ---")
        print([attr for attr in dir(sam) if not attr.startswith('__')])


    print("\n--- C. Plotting Comparison ---")
    img = Image.open(IMAGE_PATH)
    img_width, img_height = img.size
    fig, ax = plt.subplots(1, figsize=(12, 12))
    ax.imshow(img)
    ax.set_title(f"Comparison: Predicted (Blue) vs. Ground Truth (Red) for '{TEXT_PROMPT}'")

    # Helper function to convert normalized [x1, y1, x2, y2] to Matplotlib [x, y, width, height]
    def denormalize_box(norm_coords, width, height):
        x_min, y_min, x_max, y_max = norm_coords
        x = x_min * width
        y = y_min * height
        w = (x_max - x_min) * width
        h = (y_max - y_min) * height
        return (x, y, w, h)

    # 1. Plot GROUND TRUTH BOX (Red)
    gt_x, gt_y, gt_w, gt_h = denormalize_box(gt_coords_norm, img_width, img_height)
    gt_rect = patches.Rectangle((gt_x, gt_y), gt_w, gt_h,
                                linewidth=3, edgecolor='red',
                                facecolor='none', label='Ground Truth')
    ax.add_patch(gt_rect)
    plt.legend()
    plt.axis('off')
    plt.show()

# Execute the main function
plot_boxes_comparison_revised_fixed(boxes_np)

In [ ]:
def plot_and_evaluate_boxes(IMAGE_PATH, JSON_PATH, TEXT_PROMPT, boxes_np):
    # -------------------------------
    # A. Load Ground Truth Box
    # -------------------------------
    print("\n=== A. Loading Ground Truth ===")

    with open(JSON_PATH, 'r') as f:
        data = json.load(f)

    gt_norm = data["objects"][0]["obj_coord"]   # [x1, y1, x2, y2] normalized
    print(f"Ground Truth (normalized): {gt_norm}")

    # -------------------------------
    # B. IOU FUNCTION
    # -------------------------------
    def compute_iou(boxA, boxB):
        """
        Boxes must be in [x1, y1, x2, y2] absolute pixel format.
        """
        xA = max(boxA[0], boxB[0])
        yA = max(boxA[1], boxB[1])
        xB = min(boxA[2], boxB[2])
        yB = min(boxA[3], boxB[3])

        interW = max(0, xB - xA)
        interH = max(0, yB - yA)
        interArea = interW * interH

        areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
        areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

        union = areaA + areaB - interArea
        if union == 0:
            return 0

        return interArea / union

    # -------------------------------
    # C. Convert GT to absolute pixel format
    # -------------------------------
    img = Image.open(IMAGE_PATH)
    W, H = img.size

    def denorm_to_abs(norm):
        x1 = norm[0] * W
        y1 = norm[1] * H
        x2 = norm[2] * W
        y2 = norm[3] * H
        return [x1, y1, x2, y2]

    gt_abs = denorm_to_abs(gt_norm)
    print(f"Ground Truth (absolute): {gt_abs}")

    # -------------------------------
    # D. Compute IoU for all predicted boxes
    # -------------------------------
    print("\n=== B. Computing IoU with Predicted Boxes ===")
    all_ious = []

    for i, pred in enumerate(boxes_np):
        pred_abs = pred.tolist()
        iou_val = compute_iou(pred_abs, gt_abs)
        all_ious.append(iou_val)
        print(f"Predicted box {i+1}: {pred_abs} → IoU = {iou_val:.4f}")

    best_iou = max(all_ious) if len(all_ious) > 0 else 0
    best_idx = np.argmax(all_ious) if len(all_ious) > 0 else -1

    print("\n=== Best Match ===")
    print(f"Best IoU: {best_iou:.4f} (Predicted box {best_idx+1 if best_idx>=0 else 'N/A'})")

    # -------------------------------
    # E. Plot Image + Boxes
    # -------------------------------
    print("\n=== C. Plotting Ground Truth & Predicted Boxes ===")

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(img)
    ax.set_title(f"GT (Red) vs Predicted (Blue) — Prompt: \"{TEXT_PROMPT}\"")

    # Plot GT
    gt_w = gt_abs[2] - gt_abs[0]
    gt_h = gt_abs[3] - gt_abs[1]
    ax.add_patch(
        patches.Rectangle(
            (gt_abs[0], gt_abs[1]), gt_w, gt_h,
            linewidth=3, edgecolor='red', facecolor='none', label="Ground Truth"
        )
    )

    # Plot predicted boxes
    for i, pred in enumerate(boxes_np):
        x1, y1, x2, y2 = pred
        w = x2 - x1
        h = y2 - y1
        ax.add_patch(
            patches.Rectangle(
                (x1, y1), w, h,
                linewidth=2, edgecolor='blue', facecolor='none',
                label="Predicted" if i == 0 else None
            )
        )
        ax.text(x1, y1-5, f"IoU={all_ious[i]:.2f}", color='yellow', fontsize=12)

    ax.legend()
    ax.axis("off")
    plt.show()

    return all_ious, best_iou

def get_best_predicted_box(sam):
    """
    Returns the best bounding box predicted by LangSAM
    based on the highest confidence score.
    """
    if sam.boxes is None or sam.scores is None:
        return None

    boxes = sam.boxes.cpu().numpy()
    scores = sam.scores.cpu().numpy()

    best_idx = scores.argmax()
    best_box = boxes[best_idx]

    print("\n=== Best predicted box (before any GT comparison) ===")
    print(f"Score = {scores[best_idx]:.4f}")
    print(f"Box   = {best_box}")

    return best_box.reshape(1, 4)   # keep shape consistent

IMAGE_PATH = "/content/drive/My Drive/Images_val_tif/P0837_0001.tif"
JSON_PATH = "/content/drive/My Drive/P0837_0001.json"

import json
with open(JSON_PATH, 'r') as f:
  data = json.load(f)
TEXT_PROMPT = object_class = data['objects'][0]['referring_sentence']

all_ious, best_iou = plot_and_evaluate_boxes(
    IMAGE_PATH=IMAGE_PATH,
    JSON_PATH=JSON_PATH,
    TEXT_PROMPT=TEXT_PROMPT,
    boxes_np=boxes_np
)


In [ ]:
import os
import json
import numpy as np

def process_folder(folder_path, json_folder_path):
    print(f"\nFound {len(os.listdir(folder_path))} files in folder: {folder_path}")
    print("----------------------------------------------------------\n")

    # Loop over all .tif files
    for filename in sorted(os.listdir(folder_path)):
        if not filename.lower().endswith(".tif"):
            continue

        image_path = os.path.join(folder_path, filename)

        # Corresponding JSON expected to have the same prefix name
        json_name = os.path.splitext(filename)[0] + ".json"
        json_path = os.path.join(json_folder_path, json_name)

        print(f"\n\n================ Processing: {filename} ================")

        # --- Check JSON existence ---
        if not os.path.exists(json_path):
            print(f"JSON missing for: {filename}, skipping.")
            continue

        # --- Load referring sentence ---
        try:
            with open(json_path, "r") as f:
                data = json.load(f)
            text_prompt = data["objects"][0]["referring_sentence"]
            print(f"Text prompt: {text_prompt}")
        except Exception as e:
            print(f"Failed to read JSON for {filename}: {e}")
            continue

        # --- Run LangSAM prediction ---
        try:
            sam = LangSAM()
            sam.predict(
                image_path,
                text_prompt,
                box_threshold=0.24,
                text_threshold=0.24
            )
        except Exception as e:
            print(f"ERROR reading image or SAM prediction failure: {e}")
            continue

        # --- Extract predicted bounding boxes ---
        try:
            if sam.boxes is None or len(sam.boxes) == 0:
                print("⚠ No boxes found by LangSAM.")
                continue

            # Convert to numpy
            boxes_np = sam.boxes.cpu().numpy()

            # Each box format is typically [x1, y1, x2, y2]
            areas = (boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])

            # Pick the largest-area predicted box
            largest_idx = np.argmax(areas)
            largest_box = boxes_np[largest_idx].reshape(1, 4)

            print(f"Selected largest predicted box (index {largest_idx}): {largest_box}")

        except Exception as e:
            print(f"Failed to extract predicted boxes: {e}")
            continue

        # --- Evaluate IoU using the single largest predicted box ---
        try:
            plot_and_evaluate_boxes(
                IMAGE_PATH=image_path,
                JSON_PATH=json_path,
                TEXT_PROMPT=text_prompt,
                boxes_np=largest_box     # only one box now
            )
        except Exception as e:
            print(f"ERROR while plotting or computing IoU: {e}")
            continue


# ---- Run ----
FOLDER = "/content/drive/My Drive/Images_val_tif_Drive"
JSON_PATH = "/content/drive/My Drive/Annotations_Val_Drive"

process_folder(FOLDER, JSON_PATH)


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image


# ============================================================
# 1. IOU FUNCTION
# ============================================================
def compute_iou(boxA, boxB):
    """
    Inputs must be [x1, y1, x2, y2] in absolute pixel format.
    """
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interW = max(0, xB - xA)
    interH = max(0, yB - yA)
    interArea = interW * interH

    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    union = areaA + areaB - interArea
    if union <= 0:
        return 0.0

    return interArea / union


# ============================================================
# 2. PLOT + SAVE VISUALIZATION
# ============================================================
def save_visualization(image_path, gt_abs, pred_box, iou_value,
                       text_prompt, output_path):
    """
    Saves a visualization showing GT and predicted boxes.
    """
    os.makedirs(output_path.parent, exist_ok=True)

    img = Image.open(image_path)
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(img)
    ax.set_title(f"GT (Red) vs Predicted (Blue)\nPrompt: '{text_prompt}'   IoU={iou_value:.3f}")

    # ---- GT box ----
    gt_w = gt_abs[2] - gt_abs[0]
    gt_h = gt_abs[3] - gt_abs[1]
    ax.add_patch(
        patches.Rectangle(
            (gt_abs[0], gt_abs[1]), gt_w, gt_h,
            linewidth=3, edgecolor='red', facecolor='none', label="Ground Truth"
        )
    )

    # ---- Predicted box ----
    x1, y1, x2, y2 = pred_box
    w = x2 - x1
    h = y2 - y1
    ax.add_patch(
        patches.Rectangle(
            (x1, y1), w, h,
            linewidth=3, edgecolor='blue', facecolor='none', label="Predicted"
        )
    )
    ax.text(x1, y1 - 8, f"IoU={iou_value:.3f}", color='yellow', fontsize=12)

    ax.legend()
    ax.axis("off")

    fig.savefig(str(output_path), dpi=150)
    plt.close(fig)


# ============================================================
# 3. SINGLE IMAGE EVALUATION
# ============================================================
def evaluate_single_image(image_path, json_path, text_prompt, pred_box, save_dir):
    """
    Evaluates IoU + saves visualization + returns metrics dict.
    """
    img = Image.open(image_path)
    W, H = img.size

    # ---- Load GT ----
    with open(json_path, 'r') as f:
        data = json.load(f)

    gt_norm = data["objects"][0]["obj_coord"]  # [x1, y1, x2, y2] normalized

    # ---- Convert normalized to absolute ----
    gt_abs = [
        gt_norm[0] * W,
        gt_norm[1] * H,
        gt_norm[2] * W,
        gt_norm[3] * H,
    ]

    # ---- Compute IoU ----
    pred_abs = pred_box.tolist()
    iou = compute_iou(pred_abs, gt_abs)

    # ---- Save visualization ----
    img_name = Path(image_path).stem
    vis_path = save_dir / f"{img_name}_IoU_{iou:.3f}.png"

    save_visualization(
        image_path=image_path,
        gt_abs=gt_abs,
        pred_box=pred_abs,
        iou_value=iou,
        text_prompt=text_prompt,
        output_path=vis_path
    )

    return {
        "image": str(image_path),
        "gt_box": gt_abs,
        "pred_box": pred_abs,
        "iou": float(iou),
        "prompt": text_prompt,
        "vis_path": str(vis_path)
    }


# ============================================================
# 4. PROCESS WHOLE FOLDER
# ============================================================
def process_folder(folder_path, json_folder_path, output_root):
    folder_path = Path(folder_path)
    json_folder_path = Path(json_folder_path)
    output_root = Path(output_root)

    pass_dir = output_root / "pass"
    fail_dir = output_root / "failed"

    pass_dir.mkdir(parents=True, exist_ok=True)
    fail_dir.mkdir(parents=True, exist_ok=True)

    all_results = []
    correct = 0
    total = 0

    print(f"\nScanning folder: {folder_path}")

    for filename in sorted(folder_path.iterdir()):
        if filename.suffix.lower() != ".tif":
            continue

        image_path = filename
        json_path = json_folder_path / (filename.stem + ".json")

        print(f"\n=== Processing {filename.name} ===")

        # ---- Check JSON ----
        if not json_path.exists():
            print(f"Missing JSON for: {filename.name}")
            continue

        # ---- Load prompt ----
        try:
            with open(json_path, "r") as f:
                data = json.load(f)
            text_prompt = data["objects"][0]["referring_sentence"]
        except Exception as e:
            print(f"Error reading JSON: {e}")
            continue

        # ---- Predict using LangSAM ----
        try:
            sam = LangSAM()
            sam.predict(
                str(image_path),
                text_prompt,
                box_threshold=0.24,
                text_threshold=0.24
            )
        except Exception as e:
            print(f"LangSAM failed: {e}")
            continue

        # ---- Get predicted boxes ----
        if sam.boxes is None or len(sam.boxes) == 0:
            print("No boxes detected.")
            continue

        boxes_np = sam.boxes.cpu().numpy()

        # Choose largest-area box
        areas = (boxes_np[:, 2] - boxes_np[:, 0]) * (boxes_np[:, 3] - boxes_np[:, 1])
        best_idx = np.argmax(areas)
        best_box = boxes_np[best_idx].reshape(1, 4)

        # ---- Evaluate ----
        result = evaluate_single_image(
            image_path=str(image_path),
            json_path=str(json_path),
            text_prompt=text_prompt,
            pred_box=best_box[0],
            save_dir=pass_dir  # will change depending on IoU
        )

        # Move visualization into pass/failed folder
        total += 1
        if result["iou"] >= 0.5:
            correct += 1
            final_path = pass_dir / Path(result["vis_path"]).name
        else:
            final_path = fail_dir / Path(result["vis_path"]).name

        os.replace(result["vis_path"], final_path)
        result["vis_path"] = str(final_path)

        all_results.append(result)

    # ---- Summary ----
    accuracy = (correct / total * 100) if total else 0.0
    summary = {
        "total_images": total,
        "correct_predictions(iou>=0.5)": correct,
        "accuracy": round(accuracy, 2),
        "results": all_results
    }

    with open(output_root / "evaluation_results.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\n====================================================")
    print("EVALUATION COMPLETE")
    print("====================================================")
    print(f"Total images: {total}")
    print(f"Correct (IoU ≥ 0.5): {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Results saved to: {output_root / 'evaluation_results.json'}")
    print(f"Visualizations saved to: {output_root}")
    print("====================================================")


# ============================================================
# RUN
# ============================================================
FOLDER = "/content/drive/My Drive/Images_val_tif_Drive"
JSON_PATH = "/content/drive/My Drive/Annotations_Val_Drive"
OUTPUT_DIR = "/content/drive/My Drive/LangSAM_Eval_Output"

process_folder(FOLDER, JSON_PATH, OUTPUT_DIR)


In [ ]:
import numpy as np
import torch
from transformers import BertTokenizer, BertModel
from typing import List, Dict, Union
from shapely.geometry import Polygon
import json
import wandb
from dotenv import load_dotenv
load_dotenv()

class GeoNLIEvaluator:

    def __init__(self, model_name: str = 'bert-base-uncased',
                 spatial_resolution_m: float = None,
                 image_width: int = None,
                 image_height: int = None):

        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertModel.from_pretrained(model_name)
        self.model.eval()

        self.spatial_resolution_m = spatial_resolution_m

        self.image_width = image_width
        self.image_height = image_height

        self.weights = {
            'captioning': 0.20,
            'grounding': 0.30,
            'binary': 0.10,
            'numeric': 0.20,
            'semantic': 0.20
        }

    def log_to_wandb(self, scores: Dict[str, float], metadata: Dict[str, any] = None,
                     predictions: Dict[str, Union[str, float, List]] = None,
                     ground_truths: Dict[str, Union[str, float, List]] = None,
                     project_name: str = "GeoNLI-Evaluation", run_name: str = None):

        if wandb.run is None:
            wandb.init(project=project_name, name=run_name)

        wandb.log(scores)

        if metadata:
            wandb.config.update(metadata)
        if predictions:
            wandb.log({"predictions": predictions})
        if ground_truths:
            wandb.log({"ground_truths": ground_truths})

    def get_bert_embedding(self, text: str) -> torch.Tensor:

        inputs = self.tokenizer(text, return_tensors='pt', padding=True, truncation=True)
        with torch.no_grad():
            outputs = self.model(**inputs)

        return outputs.last_hidden_state[:, 0, :].squeeze()

    def pixels_to_meters(self, pixel_value: float) -> float:

        if self.spatial_resolution_m is None:
            raise ValueError("spatial_resolution_m not set. Cannot convert pixels to meters.")
        return pixel_value * self.spatial_resolution_m

    def meters_to_pixels(self, meter_value: float) -> float:

        if self.spatial_resolution_m is None:
            raise ValueError("spatial_resolution_m not set. Cannot convert meters to pixels.")
        return meter_value / self.spatial_resolution_m

    def pixel_area_to_square_meters(self, pixel_area: float) -> float:

        if self.spatial_resolution_m is None:
            raise ValueError("spatial_resolution_m not set. Cannot convert area.")
        return pixel_area * (self.spatial_resolution_m ** 2)

    def square_meters_to_pixel_area(self, area_m2: float) -> float:

        if self.spatial_resolution_m is None:
            raise ValueError("spatial_resolution_m not set. Cannot convert area.")
        return area_m2 / (self.spatial_resolution_m ** 2)

    def obb_pixel_to_meters(self, obb_pixels: List[float]) -> List[float]:

        if self.spatial_resolution_m is None:
            raise ValueError("spatial_resolution_m not set. Cannot convert OBB.")

        cx_m = self.pixels_to_meters(obb_pixels[0])
        cy_m = self.pixels_to_meters(obb_pixels[1])
        width_m = self.pixels_to_meters(obb_pixels[2])
        height_m = self.pixels_to_meters(obb_pixels[3])
        angle = obb_pixels[4]

        return [cx_m, cy_m, width_m, height_m, angle]

    def obb_meters_to_pixels(self, obb_meters: List[float]) -> List[float]:

        if self.spatial_resolution_m is None:
            raise ValueError("spatial_resolution_m not set. Cannot convert OBB.")

        cx_px = self.meters_to_pixels(obb_meters[0])
        cy_px = self.meters_to_pixels(obb_meters[1])
        width_px = self.meters_to_pixels(obb_meters[2])
        height_px = self.meters_to_pixels(obb_meters[3])
        angle = obb_meters[4]

        return [cx_px, cy_px, width_px, height_px, angle]

    def normalize_coordinates(self, value: float, dimension: float) -> float:

        if dimension is None or dimension == 0:
            raise ValueError("Image dimension not set or invalid.")
        return value / dimension

    def denormalize_coordinates(self, value: float, dimension: float) -> float:

        if dimension is None or dimension == 0:
            raise ValueError("Image dimension not set or invalid.")
        return value * dimension

    def obb_normalized_to_absolute(self, obb_norm: List[float]) -> List[float]:

        if self.image_width is None or self.image_height is None:
            raise ValueError("Image dimensions not set. Cannot convert normalized coordinates.")

        cx_px = self.denormalize_coordinates(obb_norm[0], self.image_width)
        cy_px = self.denormalize_coordinates(obb_norm[1], self.image_height)
        width_px = self.denormalize_coordinates(obb_norm[2], self.image_width)
        height_px = self.denormalize_coordinates(obb_norm[3], self.image_height)
        angle = obb_norm[4]

        return [cx_px, cy_px, width_px, height_px, angle]

    def obb_absolute_to_normalized(self, obb_abs: List[float]) -> List[float]:

        if self.image_width is None or self.image_height is None:
            raise ValueError("Image dimensions not set. Cannot normalize coordinates.")

        cx_norm = self.normalize_coordinates(obb_abs[0], self.image_width)
        cy_norm = self.normalize_coordinates(obb_abs[1], self.image_height)
        width_norm = self.normalize_coordinates(obb_abs[2], self.image_width)
        height_norm = self.normalize_coordinates(obb_abs[3], self.image_height)
        angle = obb_abs[4]

        return [cx_norm, cy_norm, width_norm, height_norm, angle]

    def validate_obb_bounds(self, obb: List[float], normalized: bool = False) -> bool:

        if self.image_width is None or self.image_height is None:
            return True

        cx, cy, width, height, angle = obb

        if normalized:
            if not (0 <= cx <= 1 and 0 <= cy <= 1 and
                    0 <= width <= 1 and 0 <= height <= 1):
                return False
        else:
            polygon = self.obb_to_polygon(obb)
            for vertex in polygon:
                if not (0 <= vertex[0] <= self.image_width and
                        0 <= vertex[1] <= self.image_height):
                    return False

        return True

    def cosine_similarity(self, emb1: torch.Tensor, emb2: torch.Tensor) -> float:

        return torch.nn.functional.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0)).item()

    def get_ngrams(self, tokens: List[str], n: int) -> List[str]:

        if n > len(tokens):
            return []
        return [' '.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

    def bert_bleu_score(self, candidate: str, reference: str, N: int = 4, epsilon: float = 1e-8) -> float:

        candidate_tokens = candidate.lower().split()
        reference_tokens = reference.lower().split()

        if len(candidate_tokens) == 0:
            return 0.0

        log_precisions = []

        for n in range(1, N + 1):

            candidate_ngrams = self.get_ngrams(candidate_tokens, n)
            reference_ngrams = self.get_ngrams(reference_tokens, n)

            if len(candidate_ngrams) == 0:
                log_precisions.append(np.log(epsilon))
                continue

            if len(reference_ngrams) == 0:
                log_precisions.append(np.log(epsilon))
                continue

            total_similarity = 0.0
            for c_ngram in candidate_ngrams:
                max_sim = 0.0
                c_emb = self.get_bert_embedding(c_ngram)

                for r_ngram in reference_ngrams:
                    r_emb = self.get_bert_embedding(r_ngram)
                    sim = self.cosine_similarity(c_emb, r_emb)
                    max_sim = max(max_sim, sim)

                total_similarity += max_sim

            precision_n = total_similarity / len(candidate_ngrams)
            log_precisions.append(np.log(precision_n + epsilon))

        bert_bleu = np.exp(np.mean(log_precisions))

        return float(np.clip(bert_bleu, 0, 1))

    def evaluate_captioning(self, candidate: str, reference: str) -> float:

        return self.bert_bleu_score(candidate, reference, N=4)

    def polygon_area(self, vertices: np.ndarray) -> float:

        x = vertices[:, 0]
        y = vertices[:, 1]
        return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

    def polygon_intersection(self, poly1: np.ndarray, poly2: np.ndarray) -> float:

        try:
            p1 = Polygon(poly1)
            p2 = Polygon(poly2)

            if not p1.is_valid:
                p1 = p1.buffer(0)
            if not p2.is_valid:
                p2 = p2.buffer(0)

            intersection = p1.intersection(p2)
            return intersection.area
        except:
            return 0.0

    def obb_to_polygon(self, obb: List[float], angle_in_degrees: bool = True) -> np.ndarray:

        cx, cy, w, h, angle = obb

        if angle_in_degrees:
            angle = np.radians(angle)

        w_half = w / 2
        h_half = h / 2

        corners = np.array([
            [-w_half, -h_half],
            [w_half, -h_half],
            [w_half, h_half],
            [-w_half, h_half]
        ])

        cos_a = np.cos(angle)
        sin_a = np.sin(angle)
        rotation_matrix = np.array([
            [cos_a, -sin_a],
            [sin_a, cos_a]
        ])

        rotated_corners = corners @ rotation_matrix.T

        rotated_corners[:, 0] += cx
        rotated_corners[:, 1] += cy

        return rotated_corners

    def compute_iou(self, box1: List[float], box2: List[float]) -> float:

        poly1 = self.obb_to_polygon(box1)
        poly2 = self.obb_to_polygon(box2)

        area1 = self.polygon_area(poly1)
        area2 = self.polygon_area(poly2)

        if area1 == 0 or area2 == 0:
            return 0.0

        intersection_area = self.polygon_intersection(poly1, poly2)

        union_area = area1 + area2 - intersection_area

        if union_area == 0:
            return 0.0

        return intersection_area / union_area

    def evaluate_grounding(self, pred_boxes: List[List[float]],
                          gt_boxes: List[List[float]], alpha: float = 1.0,
                          coordinate_system: str = 'normalized',
                          validate_bounds: bool = False) -> float:

        if coordinate_system == 'meter':
            if self.spatial_resolution_m is None:
                raise ValueError("spatial_resolution_m required for meter coordinates")
            pred_boxes = [self.obb_meters_to_pixels(box) for box in pred_boxes]
            gt_boxes = [self.obb_meters_to_pixels(box) for box in gt_boxes]
        elif coordinate_system == 'normalized':
            if self.image_width is None or self.image_height is None:
                raise ValueError("image dimensions required for normalized coordinates")
            pred_boxes = [self.obb_normalized_to_absolute(box) for box in pred_boxes]
            gt_boxes = [self.obb_normalized_to_absolute(box) for box in gt_boxes]

        if validate_bounds:
            for i, box in enumerate(pred_boxes):
                if not self.validate_obb_bounds(box, normalized=False):
                    print(f"Warning: Predicted box {i} is out of bounds")
            for i, box in enumerate(gt_boxes):
                if not self.validate_obb_bounds(box, normalized=False):
                    print(f"Warning: Ground truth box {i} is out of bounds")

        N_pred = len(pred_boxes)
        N_ref = len(gt_boxes)

        count_penalty = np.exp(-alpha * abs(N_pred - N_ref))

        if N_pred == 0 or N_ref == 0:
            return count_penalty * 0.0

        ious = []
        used_gt = set()

        for pred_box in pred_boxes:
            max_iou = 0.0
            best_gt_idx = -1

            for gt_idx, gt_box in enumerate(gt_boxes):
                if gt_idx in used_gt:
                    continue
                iou = self.compute_iou(pred_box, gt_box)
                if iou > max_iou:
                    max_iou = iou
                    best_gt_idx = gt_idx

            if best_gt_idx >= 0:
                used_gt.add(best_gt_idx)
                ious.append(max_iou)

        mean_iou = np.mean(ious) if ious else 0.0

        grounding_score = count_penalty * mean_iou

        return float(np.clip(grounding_score, 0, 1))

    def evaluate_binary(self, prediction: str, ground_truth: str) -> float:

        pred_normalized = prediction.strip().lower()
        gt_normalized = ground_truth.strip().lower()

        return 1.0 if pred_normalized == gt_normalized else 0.0

    def evaluate_numeric(self, prediction: float, ground_truth: float,
                        unit: str = None, prediction_unit: str = None) -> float:

        if unit and prediction_unit and unit != prediction_unit:
            if self.spatial_resolution_m is None:
                raise ValueError("spatial_resolution_m required for unit conversion")

            if prediction_unit == 'pixels' and unit == 'meters':
                prediction = self.pixels_to_meters(prediction)
            elif prediction_unit == 'meters' and unit == 'pixels':
                prediction = self.meters_to_pixels(prediction)
            elif prediction_unit == 'square_pixels' and unit == 'square_meters':
                prediction = self.pixel_area_to_square_meters(prediction)
            elif prediction_unit == 'square_meters' and unit == 'square_pixels':
                prediction = self.square_meters_to_pixel_area(prediction)
            else:
                raise ValueError(f"Unsupported unit conversion: {prediction_unit} to {unit}")

        score = np.exp(-abs(prediction - ground_truth))

        return float(np.clip(score, 0, 1))

    def evaluate_semantic(self, prediction: str, ground_truth: str) -> float:
        # For semantic evaluation, use direct cosine similarity on the full text
        # This avoids issues with short texts in BLEU-like scoring
        pred_emb = self.get_bert_embedding(prediction.lower())
        gt_emb = self.get_bert_embedding(ground_truth.lower())
        similarity = self.cosine_similarity(pred_emb, gt_emb)
        return float(np.clip(similarity, 0, 1))

    def compute_final_score(self, scores: Dict[str, float]) -> float:

        final_score = 0.0

        for task, weight in self.weights.items():
            if task in scores:
                final_score += weight * scores[task]

        return float(np.clip(final_score, 0, 1))

    def parse_response_json(self, response_json: dict) -> (Dict[str, Union[str, float, List]], Dict[str, any]):

        metadata = response_json.get('input_image', {}).get('metadata', {})

        self.image_width = metadata.get('width')
        self.image_height = metadata.get('height')
        self.spatial_resolution_m = metadata.get('spatial_resolution_m')

        predictions = {}

        queries = response_json.get('queries', {})

        if 'caption_query' in queries:
            predictions['captioning'] = queries['caption_query'].get('response', '')

        if 'grounding_query' in queries:
            grounding_response = queries['grounding_query'].get('response', [])
            predictions['grounding'] = [item.get('obbox', []) for item in grounding_response]


        attr = queries.get('attribute_query', {})

        if 'binary' in attr:
            predictions['binary'] = attr['binary'].get('response', '')

        if 'numeric' in attr:
            predictions['numeric'] = attr['numeric'].get('response', 0.0)

        if 'semantic' in attr:
            predictions['semantic'] = attr['semantic'].get('response', '')

        return predictions, metadata

    def load_predictions_from_json(self, json_path: str) -> (Dict[str, Union[str, float, List]], Dict[str, any]):

        with open(json_path, 'r') as f:
            data = json.load(f)
        return self.parse_response_json(data)

    def evaluate_all(self, predictions: Dict[str, Union[str, float, List]],
                     ground_truths: Dict[str, Union[str, float, List]],
                     metadata: Dict[str, any] = None) -> Dict[str, float]:

        scores = {}
        metadata = metadata or {}

        if 'captioning' in predictions and 'captioning' in ground_truths:
            scores['captioning'] = self.evaluate_captioning(
                predictions['captioning'], ground_truths['captioning']
            )

        if 'grounding' in predictions and 'grounding' in ground_truths:
            scores['grounding'] = self.evaluate_grounding(
                predictions['grounding'], ground_truths['grounding'],
                coordinate_system='normalized', validate_bounds=True
            )

        if 'binary' in predictions and 'binary' in ground_truths:
            scores['binary'] = self.evaluate_binary(
                predictions['binary'], ground_truths['binary']
            )

        if 'numeric' in predictions and 'numeric' in ground_truths:
            scores['numeric'] = self.evaluate_numeric(
                predictions['numeric'], ground_truths['numeric'],
                unit=None, prediction_unit=None
            )

        if 'semantic' in predictions and 'semantic' in ground_truths:
            scores['semantic'] = self.evaluate_semantic(
                predictions['semantic'], ground_truths['semantic']
            )

        scores['final_score'] = self.compute_final_score(scores)

        return scores

def print_scores(scores, weights):
    """Helper function to print scores in a formatted way."""
    print("\nResults:")
    for task, score in scores.items():
        if task != 'final_score':
            weight = weights.get(task, 0)
            print(f"  {task.capitalize():15s}: {score:.4f} (weight: {weight:.0%})")
    print(f"  {'FINAL SCORE':15s}: {scores['final_score']:.4f}")


In [ ]:
import json
import numpy as np

class LiteGeoNLIEvaluator(GeoNLIEvaluator):
    """
    Inherits from GeoNLIEvaluator but overrides _init_
    to skip loading BERT models, as we only need the geometric
    functions (compute_iou, obb_to_polygon, etc.)
    """
    def __init__(self):
        # We strictly strictly skip super()._init_() to avoid
        # loading 'bert-base-uncased' and torch overhead.
        self.spatial_resolution_m = 1.0
        self.image_width = 1000 # Default placeholders
        self.image_height = 1000

def calculate_metrics(json_path):
    print(f"Loading results from: {json_path}")

    # Initialize the lightweight evaluator
    evaluator = LiteGeoNLIEvaluator()

    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
    except FileNotFoundError:
        print("Error: JSON file not found.")
        return

    results_list = data.get('results', [])
    total_samples = len(results_list)

    if total_samples == 0:
        print("No results found in the JSON file.")
        return

    print(f"Processing {total_samples} samples...")

    all_ious = []

    for item in results_list:
      pred_box = item["pred_box"]
      gt_box = item["gt_box"]

      pred_obb = aabb_to_obb(pred_box)
      gt_obb   = aabb_to_obb(gt_box)

      iou = evaluator.compute_iou(pred_obb, gt_obb)
      all_ious.append(iou)

    # Convert to numpy array for easy calculation
    ious_np = np.array(all_ious)

    # 1. Overall Mean IoU (Standard mIoU)
    mean_iou = np.mean(ious_np)

    # 2. Accuracy at Thresholds (Percentage of samples where IoU >= threshold)
    acc_30 = np.mean(ious_np >= 0.3) * 100
    acc_50 = np.mean(ious_np >= 0.5) * 100
    acc_70 = np.mean(ious_np >= 0.7) * 100

    # 3. Mean IoU of Positives (Average IoU considering only matches that passed the threshold)
    # Handle edge cases where no samples pass the threshold to avoid NaN
    miou_at_30_subset = np.mean(ious_np[ious_np >= 0.3]) if np.any(ious_np >= 0.3) else 0.0
    miou_at_50_subset = np.mean(ious_np[ious_np >= 0.5]) if np.any(ious_np >= 0.5) else 0.0
    miou_at_70_subset = np.mean(ious_np[ious_np >= 0.7]) if np.any(ious_np >= 0.7) else 0.0

    print("\n" + "="*50)
    print("EVALUATION REPORT")
    print("="*50)
    print(f"Total Samples Evaluated: {total_samples}")
    print(f"Overall Mean IoU:        {mean_iou:.4f}")
    print("-" * 50)

    print("METRICS BY THRESHOLD:")
    print(f"Threshold 0.3:")
    print(f"  - Accuracy (Recall):   {acc_30:.2f}%")
    print(f"  - mIoU (subset >=0.3): {miou_at_30_subset:.4f}")

    print(f"\nThreshold 0.5:")
    print(f"  - Accuracy (Recall):   {acc_50:.2f}%")
    print(f"  - mIoU (subset >=0.5): {miou_at_50_subset:.4f}")

    print(f"\nThreshold 0.7:")
    print(f"  - Accuracy (Recall):   {acc_70:.2f}%")
    print(f"  - mIoU (subset >=0.7): {miou_at_70_subset:.4f}")
    print("="*50)

def aabb_to_obb(box):
    x1, y1, x2, y2 = box
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    w = x2 - x1
    h = y2 - y1
    angle = 0.0
    return [cx, cy, w, h, angle]

filename = "/content/drive/My Drive/LangSAM_Eval_Output/evaluation_results.json"
calculate_metrics(filename)

In [ ]:
print("Test IoU:", evaluator.compute_iou([0,0,10,10,0], [0,0,10,10,0]))

In [ ]:
import os
import json
import numpy as np

def process_folder(folder_path):
    print(f"\nFound {len(os.listdir(folder_path))} files in folder: {folder_path}")
    print("----------------------------------------------------------\n")

    # Loop over all .tif files
    for filename in sorted(os.listdir(folder_path)):
        if not filename.lower().endswith(".tif"):
            continue

        image_path = os.path.join(folder_path, filename)
        json_path  = os.path.join(folder_path, filename.replace(".tif", ".json"))

        print(f"\n\n================ Processing: {filename} ================")

        # --- Check JSON existence ---
        if not os.path.exists(json_path):
            print(f"JSON missing for: {filename}, skipping.")
            continue

        # --- Load referring sentence ---
        try:
            with open(json_path, "r") as f:
                data = json.load(f)
            text_prompt = data["objects"][0]["obj_cls"]
            print(f"Text prompt: {text_prompt}")
        except Exception as e:
            print(f"Failed to read JSON for {filename}: {e}")
            continue

        # --- Run LangSAM prediction ---
        try:
            sam = LangSAM()
            sam.predict(
                image_path,
                text_prompt,
                box_threshold=0.24,
                text_threshold=0.24
            )
        except Exception as e:
            print(f"ERROR reading image or during SAM prediction: {e}")
            print("➡ Skipping this file due to rasterio failure.")
            continue

        # --- Extract predicted bounding boxes ---
        try:
            if sam.boxes is None:
                print("⚠ No boxes found by LangSAM.")
                continue

            boxes_np = sam.boxes.cpu().numpy()
        except Exception as e:
            print(f"Failed to extract predicted boxes: {e}")
            continue

        # --- Now run your combined GT vs Prediction IoU comparison ---
        try:
            plot_and_evaluate_boxes(
                IMAGE_PATH=image_path,
                JSON_PATH=json_path,
                TEXT_PROMPT=text_prompt,
                boxes_np=boxes_np
            )
        except Exception as e:
            print(f"ERROR while plotting or computing IoU: {e}")
            continue
FOLDER = "/content/drive/My Drive/Images_val_tif"
process_folder(FOLDER)


In [ ]:
sam = LangSAM()

## Specify text prompts

In [ ]:
text_prompt = "red playground"

## Segment the image

Part of the model prediction includes setting appropriate thresholds for object detection and text association with the detected objects. These threshold values range from 0 to 1 and are set while calling the predict method of the LangSAM class.

`box_threshold`: This value is used for object detection in the image. A higher value makes the model more selective, identifying only the most confident object instances, leading to fewer overall detections. A lower value, conversely, makes the model more tolerant, leading to increased detections, including potentially less confident ones.

`text_threshold`: This value is used to associate the detected objects with the provided text prompt. A higher value requires a stronger association between the object and the text prompt, leading to more precise but potentially fewer associations. A lower value allows for looser associations, which could increase the number of associations but also introduce less precise matches.

Remember to test different threshold values on your specific data. The optimal threshold can vary depending on the quality and nature of your images, as well as the specificity of your text prompts. Make sure to choose a balance that suits your requirements, whether that's precision or recall.

In [ ]:
sam.predict(image, text_prompt, box_threshold=0.24, text_threshold=0.24)

Show the result as a grayscale image.

In [ ]:
sam.show_anns(
    cmap="Greys_r",
    add_boxes=False,
    alpha=1,
    title="Automatic Segmentation of Trees",
    blend=False,
    output="trees.tif",
)